In [ ]:
!pip install pandas openpyxl

# **Código para lidar com as o arquivos do excel RNV**




In [ ]:

import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install fuzzywuzzy
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
!pip install python-Levenshtein
import Levenshtein


In [ ]:
file_path1 = '/content/drive/MyDrive/PRIVATE_DATA/RNV_e_Natimorto_Total.xlsx'


df = pd.read_excel(file_path1)
df

In [ ]:
df['DATA NASC.'] = pd.to_datetime(df['DATA NASC.'], format='%d/%m/%Y', errors='coerce')
df = df.dropna(subset=['DATA NASC.'])
df['DATA NASC.'] = df['DATA NASC.'].dt.strftime('%d/%m/%Y')

df

In [ ]:
# Remover linhas com valores NaN na coluna "Nome"
df_combined2 = df.dropna(subset=['NOME'])

# Exibir as dimensões do DataFrame após a remoção
print("Dimensões do DataFrame após a remoção de NaN na coluna 'Nome':", df_combined2.shape)

# Exibir as primeiras linhas do DataFrame atualizado
df_combined2.shape

Retirando apenas o primeiro nome para comparação

In [ ]:
df_combined2['Primeiro_Nome'] = df_combined2['NOME'].str.split().str[0]
df_combined2['Primeiro_Nome']

In [ ]:
#Biblioteca para retirar os ç e acentos
!pip install unidecode
import unidecode


# Transformar os valores na coluna "Nome" para minúsculas e remover espaços em branco ç e acentuação e depois retirado g1,g2,g_n
df_combined2['NOME'] = df_combined2['NOME'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)
df_combined2['NOME'] = df_combined2['NOME'].str.replace(r'g[0-9]+$', '', regex=True)
df_combined2['NOME'] = df_combined2['NOME'].str.replace(r'gemelar[0-9]+$', '', regex=True)
df_combined2['Primeiro_Nome'] = df_combined2['Primeiro_Nome'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)



# Exibir as primeiras linhas do DataFrame após a transformação
df_combined2



**Para facilitar a contagem, vamos acrescentar uma coluna Gêmeo.**

In [ ]:
# Adicionando a coluna "Gêmeo"
df_combined2['Gêmeo'] = (df_combined2['NOME'] != df_combined2['NOME'].shift(1)).cumsum()

# Ajustando para 'Sim' ou 'Não'
df_combined2['Gêmeo'] = df_combined2.groupby('Gêmeo').transform('size').gt(1).map({True: 'Sim', False: 'Não'})

# Exibindo o DataFrame resultante
df_combined2

In [ ]:
df_combined2 = df_combined2[~df_combined2['NOME'].str.contains('cancelada|cancelado|desconhecida|desconhecido', case=False)]

df_combined2.shape

In [ ]:
df_gemeos = df_combined2[df_combined2['Gêmeo'] == 'Sim']

df_gemeos

In [ ]:
nome_counts = df_gemeos['NOME'].value_counts()
nome_counts

## USGs

**A partir daqui mexerei com as USGs, estou salvando cada página como uma USG**

In [ ]:
import sqlite3
db_path = '/content/drive/MyDrive/PRIVATE_DATA/USGs.db'
conn = sqlite3.connect(db_path)
df_doppler = pd.read_sql_query("SELECT * FROM Doppler", conn)
df_normal = pd.read_sql_query("SELECT * FROM Normal", conn)
df_morfologica = pd.read_sql_query("SELECT * FROM Morfologicas", conn)

# Passo 5: Feche a conexão
conn.close()

In [ ]:
# Contar valores nulos na coluna "Paciente" de cada DataFrame
nulos_doppler = df_doppler['Paciente'].isnull().sum()
nulos_normal = df_normal['Paciente'].isnull().sum()
nulos_morfologica = df_morfologica['Paciente'].isnull().sum()

nulos_doppler, nulos_normal, nulos_morfologica

In [ ]:
df_doppler = df_doppler.dropna(subset=['Paciente'])
df_normal = df_normal.dropna(subset=['Paciente'])
df_morfologica = df_morfologica.dropna(subset=['Paciente'])

In [ ]:
df_doppler.shape, df_normal.shape, df_morfologica.shape

In [ ]:
df_doppler['Primeiro_Nome'] = df_doppler['Paciente'].str.split().str[0]
df_normal['Primeiro_Nome'] = df_normal['Paciente'].str.split().str[0]
df_morfologica['Primeiro_Nome'] = df_morfologica['Paciente'].str.split().str[0]



# Transformar os valores na coluna "Paciente" para minúsculas, remover espaços em branco, ç e acentuação
df_doppler['Paciente'] = df_doppler['Paciente'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)
df_normal['Paciente'] = df_normal['Paciente'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)
df_morfologica['Paciente'] = df_morfologica['Paciente'].str.lower().str.replace(' ', '').str.replace('ç', 'c').apply(unidecode.unidecode)




In [ ]:
def clean_name(name):
    if pd.isna(name):
        return ''
    name = str(name).lower().replace('ç', 'c')
    name = unidecode.unidecode(name)
    name = ''.join(e for e in name if e.isalnum())
    return name

# Aplicar a função na coluna 'Primeiro_Nome' de cada DataFrame
df_doppler['Primeiro_Nome'] = df_doppler['Primeiro_Nome'].apply(clean_name)
df_normal['Primeiro_Nome'] = df_normal['Primeiro_Nome'].apply(clean_name)
df_morfologica['Primeiro_Nome'] = df_morfologica['Primeiro_Nome'].apply(clean_name)

df_normal

In [ ]:
# Função para remover unidades das colunas e lidar com valores None
def remove_units(df):
    df['Frequencia Cardiaca'] = df['Frequencia Cardiaca'].astype(str).str.replace('bpm', '', regex=False).str.replace(',', '.')
    df['Frequencia Cardiaca'] = pd.to_numeric(df['Frequencia Cardiaca'], errors='coerce')

    df['Peso Fetal'] = df['Peso Fetal'].astype(str).str.replace('g', '', regex=False).str.replace(',', '.')
    df['Peso Fetal'] = pd.to_numeric(df['Peso Fetal'], errors='coerce')

    df['Diametro Biparietal'] = df['Diametro Biparietal'].astype(str).str.replace('mm', '', regex=False).str.replace(',', '.')
    df['Diametro Biparietal'] = pd.to_numeric(df['Diametro Biparietal'], errors='coerce')

    df['Circunferência Cefálica'] = df['Circunferência Cefálica'].astype(str).str.replace('mm', '', regex=False).str.replace(',', '.')
    df['Circunferência Cefálica'] = pd.to_numeric(df['Circunferência Cefálica'], errors='coerce')

    # Corrigindo para 'Circunferência Abdominal' apenas no df_normal
    if 'Circunferência Abdonominal' in df.columns:
        df['Circunferência Abdominal'] = df['Circunferência Abdonominal'].astype(str).str.replace('mm', '', regex=False).str.replace(',', '.')
        df['Circunferência Abdominal'] = pd.to_numeric(df['Circunferência Abdominal'], errors='coerce')
        df.drop(columns=['Circunferência Abdonominal'], inplace=True)  # Remover a coluna antiga

    df['Comprimento Femural'] = df['Comprimento Femural'].astype(str).str.replace('mm', '', regex=False).str.replace(',', '.')
    df['Comprimento Femural'] = pd.to_numeric(df['Comprimento Femural'], errors='coerce')

    return df


# Aplicar a função de remoção de unidades nos três DataFrames
df_doppler = remove_units(df_doppler.copy())
df_normal = remove_units(df_normal.copy())
df_morfologica = remove_units(df_morfologica.copy())

# Exibir as primeiras linhas dos DataFrames após a transformação
print(df_doppler.head())
print(df_normal.head())
print(df_morfologica.head())

In [ ]:
num_unique_names = df_normal['Paciente'].nunique()
num_unique_names2 = df_doppler['Paciente'].nunique()
num_unique_names3= df_morfologica['Paciente'].nunique()
num_unique_names4 = df_combined2['NOME'].nunique()

print(num_unique_names, num_unique_names2, num_unique_names3,num_unique_names4)

## Mesclas USGs NORMAIS

**Mescla Exata do Nome**

In [ ]:
df_merged1 = pd.merge(df_combined2, df_normal, left_on='NOME', right_on='Paciente', how='inner')

df_merged1

In [ ]:
num_unique_names = df_merged1['NOME'].nunique()
num_unique_names

In [ ]:
# Criar uma cópia de df_combined2 para df_combined3
df_combined3 = df_combined2.copy()

# Identificar nomes únicos em df_combined2 e df_merged1
nomes_combined2 = set(df_combined2['NOME'])
nomes_merged1 = set(df_merged1['NOME'])

# Encontrar nomes em df_combined2 que não estão em df_merged1
nomes_exclusivos = nomes_combined2 - nomes_merged1

# Filtrar df_combined3 com os nomes exclusivos
df_combined3 = df_combined3[df_combined3['NOME'].isin(nomes_exclusivos)]

# Exibir as linhas de df_combined3
df_combined3

In [ ]:
pacientes_normal = set(df_normal['Paciente'])
pacientes_merged1 = set(df_merged1['Paciente'])

# Encontrar pacientes em df_normal que não estão em df_merged1
pacientes_exclusivos = pacientes_normal - pacientes_merged1

# Filtrar df_normal com os pacientes exclusivos
df_normal_2 = df_normal[df_normal['Paciente'].isin(pacientes_exclusivos)]

# Exibir as linhas de df_normal_exclusivos
df_normal_2

In [ ]:
# Definir a função para encontrar correspondências com score
def encontrar_correspondencia_com_score(nome, lista_nomes):
    if not isinstance(nome, str):
        return None, None  # Retorna None para ambos se o nome não for uma string

    # Certificar-se de que todos os valores em 'lista_nomes' são strings
    lista_nomes_str = lista_nomes.astype(str)

    # Encontrar as melhores correspondências usando fuzzywuzzy
    melhores_correspondencias = process.extractBests(nome, lista_nomes_str, scorer=fuzz.ratio, score_cutoff=80)

    if melhores_correspondencias:
        melhor_correspondencia = melhores_correspondencias[0][0]
        score = melhores_correspondencias[0][1]
        return melhor_correspondencia, score
    else:
        return None, None

# Aplicar a função e adicionar colunas 'Melhor_Correspondencia' e 'Indice_Similaridade'
df_combined3[['Melhor_Correspondencia', 'Indice_Similaridade']] = df_combined3['NOME'].apply(
    lambda x: pd.Series(encontrar_correspondencia_com_score(x, df_normal_2['Paciente']))
)

# Realizar o merge com base na melhor correspondência
df_merged_normal = pd.merge(df_combined3, df_normal_2, left_on='Melhor_Correspondencia', right_on='Paciente', how='inner')

# Remover a coluna 'Melhor_Correspondencia' após o merge
df_merged_normal = df_merged_normal.drop('Melhor_Correspondencia', axis=1)

In [ ]:
df_merged_normal

## USGs com Doppler

In [ ]:
df_merged2 = pd.merge(df_combined2, df_doppler, left_on='NOME', right_on='Paciente', how='inner')

df_merged2

In [ ]:
num_unique_names = df_merged2['NOME'].nunique()
num_unique_names

In [ ]:
# Criar uma cópia de df_combined2 para df_combined3
df_combined3 = df_combined2.copy()

# Identificar nomes únicos em df_combined2 e df_merged1
nomes_combined2 = set(df_combined2['NOME'])
nomes_merged2 = set(df_merged2['NOME'])

# Encontrar nomes em df_combined2 que não estão em df_merged1
nomes_exclusivos = nomes_combined2 - nomes_merged2

# Filtrar df_combined3 com os nomes exclusivos
df_combined3 = df_combined3[df_combined3['NOME'].isin(nomes_exclusivos)]

# Exibir as linhas de df_combined3
df_combined3

In [ ]:
pacientes_doppler = set(df_doppler['Paciente'])
pacientes_merged2 = set(df_merged2['Paciente'])

# Encontrar pacientes em df_normal que não estão em df_merged1
pacientes_exclusivos = pacientes_doppler - pacientes_merged2

# Filtrar df_normal com os pacientes exclusivos
df_doppler_2 = df_doppler[df_doppler['Paciente'].isin(pacientes_exclusivos)]

# Exibir as linhas de df_normal_exclusivos
df_doppler_2

In [ ]:
# Definir a função para encontrar correspondências com score
def encontrar_correspondencia_com_score(nome, lista_nomes):
    if not isinstance(nome, str):
        return None, None  # Retorna None para ambos se o nome não for uma string

    # Certificar-se de que todos os valores em 'lista_nomes' são strings
    lista_nomes_str = lista_nomes.astype(str)

    # Encontrar as melhores correspondências usando fuzzywuzzy
    melhores_correspondencias = process.extractBests(nome, lista_nomes_str, scorer=fuzz.ratio, score_cutoff=80)

    if melhores_correspondencias:
        melhor_correspondencia = melhores_correspondencias[0][0]
        score = melhores_correspondencias[0][1]
        return melhor_correspondencia, score
    else:
        return None, None

# Aplicar a função e adicionar colunas 'Melhor_Correspondencia' e 'Indice_Similaridade'
df_combined3[['Melhor_Correspondencia', 'Indice_Similaridade']] = df_combined3['NOME'].apply(
    lambda x: pd.Series(encontrar_correspondencia_com_score(x, df_doppler_2['Paciente']))
)

# Realizar o merge com base na melhor correspondência
df_merged_doppler = pd.merge(df_combined3, df_doppler_2, left_on='Melhor_Correspondencia', right_on='Paciente', how='inner')

# Remover a coluna 'Melhor_Correspondencia' após o merge
df_merged_doppler = df_merged_doppler.drop('Melhor_Correspondencia', axis=1)

#USGs Morfológicas

In [ ]:
df_merged3 = pd.merge(df_combined2, df_morfologica, left_on='NOME', right_on='Paciente', how='inner')

df_merged3

In [ ]:
num_unique_names = df_merged3['NOME'].nunique()
num_unique_names

In [ ]:
# Criar uma cópia de df_combined2 para df_combined3
df_combined3 = df_combined2.copy()

# Identificar nomes únicos em df_combined2 e df_merged1
nomes_combined2 = set(df_combined2['NOME'])
nomes_merged3 = set(df_merged3['NOME'])

# Encontrar nomes em df_combined2 que não estão em df_merged1
nomes_exclusivos = nomes_combined2 - nomes_merged3

# Filtrar df_combined3 com os nomes exclusivos
df_combined3 = df_combined3[df_combined3['NOME'].isin(nomes_exclusivos)]

# Exibir as linhas de df_combined3
df_combined3

In [ ]:
pacientes_morfologica = set(df_morfologica['Paciente'])
pacientes_merged3 = set(df_merged2['Paciente'])

# Encontrar pacientes em df_normal que não estão em df_merged1
pacientes_exclusivos = pacientes_morfologica - pacientes_merged3

# Filtrar df_normal com os pacientes exclusivos
df_morfologica_2 = df_morfologica[df_morfologica['Paciente'].isin(pacientes_exclusivos)]

# Exibir as linhas de df_normal_exclusivos
df_morfologica_2

In [ ]:
# Definir a função para encontrar correspondências com score
def encontrar_correspondencia_com_score(nome, lista_nomes):
    if not isinstance(nome, str):
        return None, None  # Retorna None para ambos se o nome não for uma string

    # Certificar-se de que todos os valores em 'lista_nomes' são strings
    lista_nomes_str = lista_nomes.astype(str)

    # Encontrar as melhores correspondências usando fuzzywuzzy
    melhores_correspondencias = process.extractBests(nome, lista_nomes_str, scorer=fuzz.ratio, score_cutoff=80)

    if melhores_correspondencias:
        melhor_correspondencia = melhores_correspondencias[0][0]
        score = melhores_correspondencias[0][1]
        return melhor_correspondencia, score
    else:
        return None, None

# Aplicar a função e adicionar colunas 'Melhor_Correspondencia' e 'Indice_Similaridade'
df_combined3[['Melhor_Correspondencia', 'Indice_Similaridade']] = df_combined3['NOME'].apply(
    lambda x: pd.Series(encontrar_correspondencia_com_score(x, df_morfologica_2['Paciente']))
)

# Realizar o merge com base na melhor correspondência
df_merged_morfologica = pd.merge(df_combined3, df_morfologica_2, left_on='Melhor_Correspondencia', right_on='Paciente', how='inner')

# Remover a coluna 'Melhor_Correspondencia' após o merge
df_merged_morfologica = df_merged_morfologica.drop('Melhor_Correspondencia', axis=1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sqlite3

# Definir o caminho para o banco de dados SQLite no Google Drive
db_path = '/content/drive/MyDrive/PRIVATE_DATA/dataframes_merge.db'

# Conectar ao banco de dados SQLite (ou criar se não existir)
conn = sqlite3.connect(db_path)

# Escrever cada dataframe em uma tabela separada
df_merged_morfologica.to_sql('Morfologica', conn, index=False, if_exists='replace')
df_merged_doppler.to_sql('Doppler', conn, index=False, if_exists='replace')
df_merged_normal.to_sql('Normal', conn, index=False, if_exists='replace')
df_merged1.to_sql('Normal_Exata', conn, index=False, if_exists='replace')
df_merged2.to_sql('Doppler_Exata', conn, index=False, if_exists='replace')
df_merged3.to_sql('Morfologica_Exata', conn, index=False, if_exists='replace')

# Fechar a conexão
conn.close()

print(f'Banco de dados SQLite salvo em: {db_path}')
